# Data Science Tools and Ecosystem

In this notebook, Data Science Tools and Ecosystem are summarized.


**Objectives**: 
* List popular languages for Data Science
* List commonly used libraries for Data Science
* evaluate arithmetic expressions

Some of the popular languages that Data Scientists use are:
1. R
2. Python
3. SQL

Some of the commonly used libraries used by Data Scientists include:
1. pandas
2. scikit-learn
3. caret

| Data Science Tools        |
|-------------|
| Jupyter Notebooks       |
| RStudio      |
| VS Code      |


### Below are a few examples of evaluating arithmetic expressions in Python.

In [19]:
#This a simple arithmetic expression to mutiply then add integers
result=(3*4)+5
result

17

In [20]:
#This will convert 200 minutes to hours by diving by 60
conversion= round((200/60),2)
conversion

3.33

## Author
Lukasz Laskowski

In [7]:
import numpy as np
a=np.array([[1,1,1],[1,1,1]]) 
print(a)

[[1 1 1]
 [1 1 1]]


# RUN SQL Server from MAC

Pull the SQL Server image:
docker pull mcr.microsoft.com/mssql/server:2022-latest

Run the SQL Server container:
docker run -e "ACCEPT_EULA=Y" -e "SA_PASSWORD=YourPassword123" -p 1433:1433 --name sqlserver -d mcr.microsoft.com/mssql/server:2022-latest

Verify the container is running:
docker ps

When you run your Docker container, use the -v flag to map the SQL Server data directory to a path on your Mac:

docker run -e "ACCEPT_EULA=Y" -e "SA_PASSWORD=YourPassword123" \
-p 1433:1433 --name sqlserver \
-v /Users/lukaszlaskowski/Documents:/var/opt/mssql \
-d mcr.microsoft.com/mssql/server:2022-latest

Use sqlite3 to connect to import csv, connect to a new db "myFreeDB.db" in current directory, use that csv to create a table in that db. Then use panda's dataframe df to select all columns in sql syntax.

SQLite3 - in-process Python library that incorporates self-contained, serverless zero-configurationn, transactional SQL database engine

In [6]:
import pandas as pd
import sqlite3
 
data = pd.read_csv('/Users/lukaszlaskowski/Desktop/Project3SQL/TableSQLLite.csv')
conn = sqlite3.connect('myFreeDB.db')
data.to_sql('TableSQLLiteTest1', conn)

df= pd.read_sql ("SELECT * FROM TableSQLLiteTest1", conn)
print(df)

   index  ID Name
0      0  10  Luk
1      1  11  Mon
2      2  12  Ebi


In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[ ]:


import dash
import more_itertools
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px

# Load the data using pandas
data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv')

# Initialize the Dash app
app = dash.Dash(__name__)

# Set the title of the dashboard
#app.title = "Automobile Statistics Dashboard"

#---------------------------------------------------------------------------------
# Create the dropdown menu options
dropdown_options = [
    {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
]
# List of years 
year_list = [i for i in range(1980, 2024, 1)]
#---------------------------------------------------------------------------------------
# Create the layout of the app
app.layout = html.Div([
    #TASK 2.1 Add title to the dashboard
    html.H1("Automobile Sales Statistics Dashboard", style={'textAlign': 'center', 'color': ' #503D36','font-size': 24}) #Include style for title
    #TASK 2.2: Add two dropdown menus
    html.Div([
        html.Label("Select Statistics:"),
         dcc.Dropdown(id='dropdown-statistics', 
                   options=[
                           {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
                           {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
                           ],
                  placeholder='Select a report type',
                  value='Select Statistics')
    ]),
    html.Div(dcc.Dropdown(
            id='select-year',
            options=[{'label': i, 'value': i} for i in year_list],
            value='Select-year',
            placeholder='Select-year')
        )),
    html.Div([#TASK 2.3: Add a division for output display
    html.Div(id='output-container', className='chart-grid', style={'display':'flex'}),])

#TASK 2.4: Creating Callbacks
# Define the callback function to update the input container based on the selected statistics
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics',component_property='value'))

def update_input_container(selected_statistics):
    if selected_statistics =='Yearly Statistics': 
        return False
    else: 
        return True

#Callback for plotting
# Define the callback function to update the input container based on the selected statistics
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [Input(component_id='dropdown-statistics', component_property='value'), Input(component_id='select-year', component_property='value')])


def update_output_container(selected_statistics, input_year):
    if selected_statistics == 'Recession Period Statistics':
        # Filter the data for recession periods
        recession_data = data[data['Recession'] == 1]
        
#TASK 2.5: Create and display graphs for Recession Report Statistics

#Plot 1 Automobile sales fluctuate over Recession Period (year wise)
        # use groupby to create relevant data for plotting
        yearly_rec=recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(
            figure=px.line(yearly_rec, 
                x='Year',
                y='Automobile_Sales',
                title="Average Automobile Sales fluctuation over Recession Period"))

#Plot 2 Calculate the average number of vehicles sold by vehicle type       
        
        # use groupby to create relevant data for plotting
        #Hint:Use Vehicle_Type and Automobile_Sales columns
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()                
        R_chart2  = dcc.Graph(
            figure=px.bar(average_sales,
            x='Vehicle_Type',
            y='Automobile_Sales',
            title="Avg Vehicles Sold in Recession"))
        
# Plot 3 Pie chart for total expenditure share by vehicle type during recessions
        # grouping data for plotting
	# Hint:Use Vehicle_Type and Advertising_Expenditure columns
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(
            figure=px.pie(exp_rec,
                  values='Advertising_Expenditure',
                  names='Vehicle_Type',
                  title="Total Ads Expenditure by Vehicle Type in Recession")
        )
# Plot 4 bar chart for the effect of unemployment rate on vehicle type and sales
        #grouping data for plotting
	# Hint:Use unemployment_rate,Vehicle_Type and Automobile_Sales columns
        unemp_data = recession_data.groupby(['Unemployment_Rate', 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(
            figure=px.bar(unemp_data,
                  x='Unemployment_Rate',
                  y='Automobile_Sales',
                  color='Vehicle_Type',
                  labels={'Unemployment_Rate': 'Unemployment Rate', 'Automobile_Sales': 'Average Automobile Sales'},
                  title='Effect of Unemployment Rate on Vehicle Type and Sales')
        )


        return [
            html.Div(className='chart-item', children=[html.Div(children=R_chart1), html.Div(children=R_chart2)], style={'display': 'flex'}),
            html.Div(className='chart-item', children=[html.Div(children=R_chart3), html.Div(children=R_chart4)], style={'display': 'flex'})
        ]

# TASK 2.6: Create and display graphs for Yearly Report Statistics
 # Yearly Statistic Report Plots
    # Check for Yearly Statistics.                             
elif (input_year and selected_statistics == 'Yearly Statistics'):
    yearly_data = data[data['Year'] == input_year]

    # Plot 1: Yearly Automobile sales using line chart for the whole period.
    # Grouping data for plotting.
    # Hint: Use the columns Year and Automobile_Sales.
    yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()
    Y_chart1 = dcc.Graph(figure=px.line(
        yas,
        x='Year',
        y='Automobile_Sales',
        title='Yearly Automobile Sales for the Whole Period'
    ))

    # Plot 2: Total Monthly Automobile sales using line chart.
    # Grouping data for plotting.
    # Hint: Use the columns Month and Automobile_Sales.
    mas = data.groupby('Month')['Automobile_Sales'].sum().reset_index()
    Y_chart2 = dcc.Graph(figure=px.line(
        mas,
        x='Month',
        y='Automobile_Sales',
        title='Total Monthly Automobile Sales'
    ))

    # Plot bar chart for average number of vehicles sold during the given year.
    # Grouping data for plotting.
    # Hint: Use the columns Year and Automobile_Sales.
    avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
    Y_chart3 = dcc.Graph(
        figure=px.bar(
            avr_vdata,
            x='Vehicle_Type',
            y='Automobile_Sales',
            title='Average Vehicles Sold by Vehicle Type in the Year {}'.format(input_year)
        )
    )

    # Plot 4: Total Advertisement Expenditure for each vehicle using pie chart.
    # Grouping data for plotting.
    # Hint: Use the columns Vehicle_Type and Advertising_Expenditure.
    exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
    Y_chart4 = dcc.Graph(
        figure=px.pie(
            exp_data,
            values='Advertising_Expenditure',
            names='Vehicle_Type',
            title='Total Advertisement Expenditure for Each Vehicle in the Year {}'.format(input_year)
        )
    )

    return [
        html.Div(className='chart-item', children=[html.Div(children=Y_chart1), html.Div(children=Y_chart2)], style={'display': 'flex'}),
        html.Div(className='chart-item', children=[html.Div(children=Y_chart3), html.Div(children=Y_chart4)], style={'display': 'flex'})
    ]
        
    else:
        return None

# Run the Dash app
if __name__ == '__main__':
    app.run_server(debug=True)